# E1 6단계: Hailo DFC 최적화 레벨 2 (GPU) vs 레벨 0 — RN50 이미지 인코더
**목적**: 노트북(GPU 없음, 레벨 0)에서 8비트 구성이 무너진 원인이 'GPU 최적화가 꺼져서'인지 확인한다.
바꾸는 것은 `optimization_level` 하나뿐이다(0 → 2). 모델(parse된 .har), 보정 사진 64장, DFC 5.4.0, 평가(Pets dev 990장)는 노트북과 같다.

**실행 전 설정 (오른쪽 패널)**
1. Accelerator: **GPU T4 x2** (P100은 쓰지 않음)
2. Internet: **On** (전화번호 인증 필요)
3. Add Input → 비공개 데이터셋 `kaggle_upload` 추가

**순서**: 위에서부터 셀을 하나씩 실행. ③에서 GPU가 안 보이면 거기서 멈추고 결과를 Claude에게 전달.
결과는 `/kaggle/working/` (emu_eval.jsonl, summary.csv, logs/, env/) → 오른쪽 Output에서 내려받기.

## ① 입력 파일 찾기

In [ ]:
import glob, os, re, json, time, subprocess, shutil
def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, executable='/bin/bash', capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-4000:])
    if check and r.returncode: raise RuntimeError(f'실패(exit {r.returncode}): {cmd[:120]}')
    return r.returncode
cands = glob.glob('/kaggle/input/**/rn50_parsed.har', recursive=True)
assert cands, '데이터셋을 못 찾음: 오른쪽 Add Input으로 kaggle_upload 데이터셋을 추가했는지 확인'
INP = os.path.dirname(cands[0])
whl = glob.glob('/kaggle/input/**/hailo_dataflow_compiler-5.4.0*.whl*', recursive=True)
assert whl, 'DFC whl 파일이 없음'
WHL = '/tmp/hailo_dataflow_compiler-5.4.0-py3-none-linux_x86_64.whl'
if os.path.isdir(whl[0]):
    raise RuntimeError('Kaggle이 whl을 압축 해제했음 → Claude에게 알려주세요(이름을 바꿔 다시 올려야 함)')
shutil.copy(whl[0], WHL)
WORK, RUNS, VENV = '/kaggle/working', '/tmp/runs', '/tmp/hv'
for d in [f'{WORK}/logs', f'{WORK}/env', RUNS]: os.makedirs(d, exist_ok=True)
print('INP =', INP); print(sorted(os.listdir(INP))); print(sorted(os.listdir(f'{INP}/alls')))
sh(f'cd {INP} && sha256sum -c SHA256SUMS.txt 2>&1 | sed "s#.*/##"')   # 노트북에서 올린 파일과 같은지

## ② 설치 (약 5~10분): 노트북과 같은 Python 3.10 + 같은 패키지 버전 + DFC 5.4.0

In [ ]:
sh('nvidia-smi | tee /kaggle/working/env/nvidia_smi.txt')
sh('apt-get -qq update > /dev/null && apt-get -qq install -y graphviz graphviz-dev libgraphviz-dev > /dev/null && echo apt OK', check=True)
sh('pip -q install uv && uv --version', check=True)
sh(f'uv venv {VENV} --python 3.10 -q && {VENV}/bin/python --version', check=True)
# 노트북 venv의 pip freeze에서 DFC 줄과 로컬 경로 줄만 빼고 같은 버전으로 설치
sh(f"grep -v -e '^hailo-dataflow-compiler' -e ' @ file:' {INP}/laptop_hailo10_venv_freeze.txt > /tmp/req.txt")
sh(f'VIRTUAL_ENV={VENV} uv pip install -q -r /tmp/req.txt && echo req OK', check=True)
sh(f'VIRTUAL_ENV={VENV} uv pip install -q --no-deps {WHL} && echo DFC OK', check=True)
sh(f'{VENV}/bin/hailo --version; VIRTUAL_ENV={VENV} uv pip freeze > {WORK}/env/kaggle_freeze.txt')

## ③ GPU 인식 확인 — 여기서 GPU가 0개면 중단

In [ ]:
sh(f'''{VENV}/bin/python -c "
import tensorflow as tf
g = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPU', len(g), g)
print('CUDA build', tf.sysconfig.get_build_info().get('cuda_version'), '| cuDNN build', tf.sysconfig.get_build_info().get('cudnn_version'))
" 2>&1 | grep -v -e "^W0000" -e "^I0000" | tee {WORK}/env/tf_gpu.txt''')
print('\n→ 마지막 줄 위에 "GPU 1" 이상이면 통과. "GPU 0"이면 이 셀 출력을 Claude에게 전달하고 멈추세요.')

## ④ 실행 함수: 양자화(optimize) → 로그 점검 → 에뮬레이터 990장 평가

In [ ]:
def run(cfg, lv, rep=1):
    tag = f'{cfg}_lv{lv}' + (f'_r{rep}' if rep > 1 else '')
    d = f'{RUNS}/{tag}'; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    log = f'{WORK}/logs/{tag}_optimize.log'
    t = time.time()
    rc = sh(f'cd {d} && {VENV}/bin/hailo optimize {INP}/rn50_parsed.har --hw-arch hailo10h '
            f'--calib-set-path {INP}/calib_rn50_highres64_s0_nhwc_u8.npy '
            f'--model-script {INP}/alls/{cfg}_lv{lv}.alls > {log} 2>&1')
    opt_sec = round(time.time() - t)
    txt = open(log, errors='ignore').read()
    algos = sorted(set(m.strip() for m in re.findall(r'Starting ([A-Z][A-Za-z ]+?)(?:\s*$|\s*\()', txt, re.M)))
    chk = {'tag': tag, 'optimize_exit': rc, 'opt_sec': opt_sec,
           'lowered_to_lv0': 'Reducing optimization level' in txt,
           'no_gpu_warning': 'no available GPU' in txt, 'algorithms': algos}
    har = glob.glob(f'{d}/*_optimized.har')
    if har:
        t = time.time()
        sh(f'cd {d} && {VENV}/bin/python {INP}/emu_eval.py --har {har[0]} --ctx quantized --data pets_dev --n 990 '
           f'--tag kaggle_{tag} --cache {INP} --out {WORK}/emu_eval.jsonl > {WORK}/logs/{tag}_eval.log 2>&1')
        chk['eval_sec'] = round(time.time() - t)
        rows = [json.loads(l) for l in open(f'{WORK}/emu_eval.jsonl') if f'"kaggle_{tag}"' in l]
        if rows: chk.update({k: rows[-1][k] for k in ('top1', 'top5', 'cos_mean', 'cos_p1')})
    with open(f'{WORK}/run_checks.jsonl', 'a') as f: f.write(json.dumps(chk, ensure_ascii=False) + '\n')
    print(json.dumps(chk, ensure_ascii=False))
    return chk

## ⑤ 대조 실험: 레벨 0 공식 설정 (약 5분)
노트북 결과(top-1 **81.52**, cos 0.9346)와 거의 같아야 Kaggle 환경을 믿을 수 있다.

In [ ]:
c = run('official', 0)
print('노트북과 차이(top-1, %p):', None if 'top1' not in c else round(c['top1'] - 81.52, 2))

## ⑥ 레벨 2 실험 (약 40~70분)
각 줄의 `lowered_to_lv0`가 **False**이고 `algorithms`에 Fine Tune 등이 보여야 '레벨 2가 실제로 켜졌다'고 판정한다.

In [ ]:
for cfg in ['official', 'off8L4', 'base8', 'off8L2L3', 'c3eq']:
    run(cfg, 2)
run('c3eq', 2, rep=2)   # 같은 설정을 한 번 더: 미세조정의 무작위성 확인

## ⑦ 요약표 (노트북 레벨 0 결과와 나란히)

In [ ]:
LV0_LAPTOP = {'official': 81.52, 'off8L4': 7.07, 'base8': 3.33, 'off8L2L3': 62.93, 'c3eq': 4.75}  # Pets dev 990장, top-1
FP = 84.34
import csv
rows = [json.loads(l) for l in open(f'{WORK}/run_checks.jsonl')]
with open(f'{WORK}/summary.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['tag', 'lv0_laptop_top1', 'top1', 'drop_vs_fp', 'top5', 'cos_mean', 'lowered_to_lv0', 'algorithms', 'opt_sec'])
    for r in rows:
        cfg = r['tag'].split('_lv')[0]
        w.writerow([r['tag'], LV0_LAPTOP.get(cfg), r.get('top1'), None if r.get('top1') is None else round(r['top1'] - FP, 2),
                    r.get('top5'), r.get('cos_mean'), r['lowered_to_lv0'], ';'.join(r['algorithms']), r['opt_sec']])
print(open(f'{WORK}/summary.csv').read())
sh(f'cd {WORK} && tar czf logs.tgz logs env && ls -la')